In [1]:
#Verify CUDA toolchain
!nvidia-smi
!nvcc --version

Thu Oct 16 09:43:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#Standard C headers and CUDA runtime needed by host code and kernel

%%writefile main.cu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cuda_runtime.h>

// -------- Launch configurables (override at compile time with -DGRID_BLOCKS, -DGRID_THREADS)
#ifndef GRID_BLOCKS
#define GRID_BLOCKS 32
#endif
#ifndef GRID_THREADS
#define GRID_THREADS 128
#endif

// -------- Helpers: wrap ranges
__device__ __forceinline__ char wrapLower(char c) {
	if (c > 122) return char((c - 122) + 97);
	if (c < 97)  return char((97 - c) + 97);
	return c;
}
__device__ __forceinline__ char wrapDigit(char c) {
	if (c > 57) return char((c - 57) + 48);
	if (c < 48) return char((48 - c) + 48);
	return c;
}

// -------- Device crypt: mirrors the provided cudaCrypt on CPU
__device__ void deviceCrypt(const char* rawPassword, char* newPassword) {
	newPassword[0] = wrapLower(rawPassword[0] + 3);
	newPassword[1] = wrapLower(rawPassword[0] - 2);
	newPassword[2] = wrapLower(rawPassword[0] + 1);
	newPassword[3] = wrapLower(rawPassword[1] + 1);
	newPassword[4] = wrapLower(rawPassword[1] - 2);
	newPassword[5] = wrapLower(rawPassword[1] - 3);
	newPassword[6] = wrapDigit(rawPassword[2] + 1);
	newPassword[7] = wrapDigit(rawPassword[2] - 2);
	newPassword[8] = wrapDigit(rawPassword[3] + 4);
	newPassword[9] = wrapDigit(rawPassword[3] - 3);
	newPassword[10] = '\0';
}

__device__ __forceinline__ bool match10(const char* a, const char* b) {
	#pragma unroll
	for (int i = 0; i < 10; ++i) {
		if (a[i] != b[i]) return false;
	}
	return true;
}

// -------- Kernel: grid-stride; generates candidates, encrypts on device, compares, returns result
__global__ void crackKernel(const char* targetEncrypted10, char* foundRaw4, int* foundFlag) {
	const long long total = 26LL * 26LL * 10LL * 10LL; // 67,600
	const long long stride = (long long)gridDim.x * (long long)blockDim.x;
	long long idx = (long long)blockIdx.x * (long long)blockDim.x + (long long)threadIdx.x;

	char raw[5]; raw[4] = '\0';
	char enc[11];

	for (; idx < total; idx += stride) {
		// quick exit if another thread already found it
		if (atomicAdd(foundFlag, 0) != 0) return;

		long long t = idx;
		int d2 = (int)(t % 10); t /= 10;   // last digit
		int d1 = (int)(t % 10); t /= 10;   // first digit
		int l2 = (int)(t % 26); t /= 26;   // second letter
		int l1 = (int)(t % 26);            // first letter

		raw[0] = char('a' + l1);
		raw[1] = char('a' + l2);
		raw[2] = char('0' + d1);
		raw[3] = char('0' + d2);

		deviceCrypt(raw, enc);

		if (match10(enc, targetEncrypted10)) {
			// claim once
			if (atomicCAS(foundFlag, 0, 1) == 0) {
				foundRaw4[0] = raw[0];
				foundRaw4[1] = raw[1];
				foundRaw4[2] = raw[2];
				foundRaw4[3] = raw[3];
			}
			return;
		}
	}
}

// -------- Host utilities
static void checkCuda(cudaError_t err, const char* what) {
	if (err != cudaSuccess) {
		fprintf(stderr, "CUDA error at %s: %s\n", what, cudaGetErrorString(err));
		std::exit(1);
	}
}

int main(int argc, char** argv) {
	if (argc != 2) {
		printf("Usage: %s <encrypted_password_10_chars>\n", argv[0]);
		return 1;
	}
	const char* targetEnc = argv[1];
	if (std::strlen(targetEnc) != 10) {
		fprintf(stderr, "Input must be exactly 10 chars of encrypted password.\n");
		return 1;
	}

	// Allocate device memory (sizes match requirements)
	char* d_target = nullptr;     // 10 + NUL for convenience
	char* d_foundRaw = nullptr;   // 4 bytes
	int* d_foundFlag = nullptr;   // 4 bytes
	checkCuda(cudaMalloc((void**)&d_target, 11), "cudaMalloc d_target");
	checkCuda(cudaMalloc((void**)&d_foundRaw, 4), "cudaMalloc d_foundRaw");
	checkCuda(cudaMalloc((void**)&d_foundFlag, sizeof(int)), "cudaMalloc d_foundFlag");

	// Initialize device buffers
	checkCuda(cudaMemcpy(d_target, targetEnc, 10, cudaMemcpyHostToDevice), "copy targetEnc");
	checkCuda(cudaMemset(d_target + 10, 0, 1), "set terminator");
	checkCuda(cudaMemset(d_foundRaw, 0, 4), "memset foundRaw");
	int zero = 0;
	checkCuda(cudaMemcpy(d_foundFlag, &zero, sizeof(int), cudaMemcpyHostToDevice), "init foundFlag");

	// Launch configuration (can be overridden at compile time)
	dim3 blocks(GRID_BLOCKS);
	dim3 threads(GRID_THREADS);

	// Timing
	cudaEvent_t start, stop;
	checkCuda(cudaEventCreate(&start), "event create start");
	checkCuda(cudaEventCreate(&stop), "event create stop");
	checkCuda(cudaEventRecord(start), "event record start");

	// Launch
	crackKernel<<<blocks, threads>>>(d_target, d_foundRaw, d_foundFlag);
	checkCuda(cudaGetLastError(), "kernel launch");
	checkCuda(cudaDeviceSynchronize(), "kernel sync");

	// Stop timing
	checkCuda(cudaEventRecord(stop), "event record stop");
	checkCuda(cudaEventSynchronize(stop), "event sync stop");
	float ms = 0.0f;
	checkCuda(cudaEventElapsedTime(&ms, start, stop), "event elapsed");
	cudaEventDestroy(start);
	cudaEventDestroy(stop);

	// Fetch results
	int found = 0;
	char rawOut[5]; rawOut[4] = '\0';
	checkCuda(cudaMemcpy(&found, d_foundFlag, sizeof(int), cudaMemcpyDeviceToHost), "copy foundFlag");
	if (found) {
		checkCuda(cudaMemcpy(rawOut, d_foundRaw, 4, cudaMemcpyDeviceToHost), "copy foundRaw");
		printf("Decrypted raw password: %s\n", rawOut);
	} else {
		printf("Password not found in search space.\n");
	}

	printf("Kernel elapsed: %.3f ms (blocks=%u, threads=%u)\n", ms, blocks.x, threads.x);

	// Free device memory
	cudaFree(d_target);
	cudaFree(d_foundRaw);
	cudaFree(d_foundFlag);
	return 0;
}

Overwriting main.cu


In [46]:
#Compile for Tesla T4 (sm_75) with a baseline grid
!/usr/local/cuda/bin/nvcc -O2 -std=c++14 -gencode=arch=compute_75,code=sm_75 -DGRID_BLOCKS=32 -DGRID_THREADS=128 main.cu -o crack

In [50]:
#Run the cracker with a known 10-character encrypted input (prints decrypted raw + timing)
!./crack kfiqnm1770

Decrypted raw password: hp93
Kernel elapsed: 0.125 ms (blocks=32, threads=128)


In [ ]:
#Alternative compile using -gencode (to keep it on one line to avoid shell parsing issues)
!/usr/local/cuda/bin/nvcc -O2 -std=c++14 \
  -gencode arch=compute_75,code=sm_75 \
  -DGRID_BLOCKS=32 -DGRID_THREADS=128 \
  main.cu -o crack

In [49]:
#Demonstrate correctness on multiple targets produced by the provided cudaCrypt rules
!./crack kfiqnm1770
!./crack dcbbcd1243
!./crack dxbbxw1746
!./crack pknnkj6392
!./crack dcbbxw1246

Decrypted raw password: hp93
Kernel elapsed: 0.129 ms (blocks=32, threads=128)
Decrypted raw password: aa00
Kernel elapsed: 0.103 ms (blocks=32, threads=128)
Decrypted raw password: zz99
Kernel elapsed: 0.121 ms (blocks=32, threads=128)
Decrypted raw password: mm55
Kernel elapsed: 0.113 ms (blocks=32, threads=128)
Decrypted raw password: az09
Kernel elapsed: 0.141 ms (blocks=32, threads=128)


In [51]:
#Rebuild with a different grid to show it works with multiple blocks/threads and compare timings
!./crack dcbbcd1243

Decrypted raw password: aa00
Kernel elapsed: 0.113 ms (blocks=32, threads=128)


In [52]:
#Build for T4 with config 1: blocks=16, threads=256
!/usr/local/cuda/bin/nvcc -O2 -std=c++14 -arch=sm_75 -code=sm_75 \
  -DGRID_BLOCKS=16 -DGRID_THREADS=256 \
  main.cu -o crack

#Run two different encrypted inputs (prints decrypted raw + Kernel elapsed ms)
!./crack dcbbcd1243
!./crack dxbbxw1746

nvcc fatal   : Value of -arch option ('sm_75') must be a virtual code architecture
Decrypted raw password: aa00
Kernel elapsed: 0.123 ms (blocks=32, threads=128)
Decrypted raw password: zz99
Kernel elapsed: 0.119 ms (blocks=32, threads=128)


In [53]:
#Build for T4 with config 2: blocks=64, threads=64 (second timing point)
!/usr/local/cuda/bin/nvcc -O2 -std=c++14 -arch=sm_75 -code=sm_75 \
  -DGRID_BLOCKS=64 -DGRID_THREADS=64 \
  main.cu -o crack

#Run the same two inputs again to compare timings across configs
!./crack dcbbcd1243
!./crack dxbbxw1746

nvcc fatal   : Value of -arch option ('sm_75') must be a virtual code architecture
Decrypted raw password: aa00
Kernel elapsed: 0.135 ms (blocks=32, threads=128)
Decrypted raw password: zz99
Kernel elapsed: 0.145 ms (blocks=32, threads=128)
